[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DavinciDreams/SymbioGPT/blob/main/compress_svd_juliaslm.ipynb)

# SVD-Guided Compression: Using Wolves Signal Without Evolution

The wolves evolutionary run on JuliaSLM showed real signal — loss improved
6.83→6.22 over 10 gens — but kept disconnecting on Colab. The wolves' core
insight is that **layers have different compressibility** (different SVD rank
spectra). Some layers can be factored to rank 40, others need rank 200+.

This notebook **skips the evolutionary search** and uses the SVD energy analysis
directly to set per-layer rank targets. Three energy thresholds give three
compression levels:

| Config | Energy Retained | Strategy |
|--------|----------------|----------|
| SVD-95 | 95% per layer | Conservative — minimal information loss |
| SVD-90 | 90% per layer | Moderate — good compression/quality tradeoff |
| SVD-80 | 80% per layer | Aggressive — maximum compression |

Each layer gets its own rank target based on its SVD spectrum, then is replaced
with `FactoredLinear(A, B)` where `W ≈ A @ B` via truncated SVD. This is
deterministic and instant — no evolution, no disconnection risk.

**Source model**: JuliaSLM (5.04M params, d=256, 6L, val_loss=3.54)

**Companion notebook**: `compress_juliaslm.ipynb` does blind uniform truncation
(d=256→192) for comparison. This notebook tests whether informed per-layer SVD
compression outperforms uniform dimension reduction.

GitHub: https://github.com/DavinciDreams/SymbioGPT

In [ ]:
# 1. Setup
!pip install -q wandb huggingface_hub
!mkdir -p /content/SymbioGPT
%cd /content/SymbioGPT

In [ ]:
# 2. GPU check
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    print(f"Memory: {mem / 1e9:.1f} GB")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 3. W&B + HF login
import wandb
from huggingface_hub import login as hf_login

wandb.login()
hf_login()

In [ ]:
# 4. Download data + JuliaSLM weights
import os, sys, math, time, copy
from dataclasses import dataclass
from typing import Dict, List, Tuple
import numpy as np
from huggingface_hub import hf_hub_download
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_REPO = "LisaMegaWatts/SymbioGPT-10M"
SLM_REPO = "LisaMegaWatts/JuliaSLM"
os.makedirs("data", exist_ok=True)

print("Downloading pre-tokenized data...")
hf_hub_download(repo_id=DATA_REPO, filename="data/train_curated.txt.tokens.pt", local_dir=".")
hf_hub_download(repo_id=DATA_REPO, filename="data/val.txt.tokens.pt", local_dir=".")

print("Downloading JuliaSLM weights (NPZ)...")
hf_hub_download(repo_id=SLM_REPO, filename="juliaslm_weights.npz", local_dir=".")

CTX = 256
print("Loading tokens...")
train_tokens = torch.load("data/train_curated.txt.tokens.pt", weights_only=True).tolist()
val_tokens = torch.load("data/val.txt.tokens.pt", weights_only=True).tolist()

def chunk(tokens, seq_len):
    n = len(tokens) // (seq_len + 1)
    tokens = tokens[:n * (seq_len + 1)]
    data = torch.tensor(tokens, dtype=torch.long).reshape(n, seq_len + 1)
    return data[:, :-1], data[:, 1:]

train_inputs, train_labels = chunk(train_tokens, CTX)
val_inputs, val_labels = chunk(val_tokens, CTX)
print(f"Train: {len(train_inputs):,} seqs ({len(train_inputs)*CTX:,} tokens)")
print(f"Val: {len(val_inputs):,} seqs")
del train_tokens, val_tokens

In [ ]:
# 5. JuliaSLM model definition
#
# IMPORTANT: Julia's column-major reshape(result, HD, T, H, B) fills dims
# HD→T→H→B (fastest→slowest). Python's row-major equivalent is
# view(B, H, T, HD) — NOT view(B, T, H, HD).transpose(1, 2).
# The model was TRAINED with Julia's layout, so we must match it exactly.

@dataclass
class JuliaSLMConfig:
    d_model: int = 256
    n_layers: int = 6
    n_heads: int = 4
    head_dim: int = 64
    ffn_inner: int = 640
    context_length: int = 256
    vocab_size: int = 2000
    weight_tying: bool = True
    rope_base: float = 10000.0


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 256, base: float = 10000.0):
        super().__init__()
        freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        positions = torch.arange(max_seq_len).float()
        angles = torch.outer(positions, freqs)
        self.register_buffer("cos_cache", angles.cos())
        self.register_buffer("sin_cache", angles.sin())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(2)
        half = x.size(-1) // 2
        x1, x2 = x[..., :half], x[..., half:]
        cos = self.cos_cache[:seq_len, :half].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cache[:seq_len, :half].unsqueeze(0).unsqueeze(0)
        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, head_dim: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        total_dim = n_heads * head_dim
        self.wq = nn.Linear(d_model, total_dim, bias=False)
        self.wk = nn.Linear(d_model, total_dim, bias=False)
        self.wv = nn.Linear(d_model, total_dim, bias=False)
        self.wo = nn.Linear(total_dim, d_model, bias=False)

    def forward(self, x: torch.Tensor, rope: RotaryEmbedding,
                mask: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        H, HD = self.n_heads, self.head_dim
        # Julia-matching reshape: view(B, H, T, HD) matches Julia's
        # column-major reshape(HD, T, H, B)
        q = self.wq(x).view(B, H, T, HD)
        k = self.wk(x).view(B, H, T, HD)
        v = self.wv(x).view(B, H, T, HD)
        q = rope(q)
        k = rope(k)
        scale = 1.0 / math.sqrt(HD)
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn = attn + mask
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        out = out.contiguous().view(B, T, H * HD)
        return self.wo(out)


class SwiGLUFFN(nn.Module):
    def __init__(self, d_model: int, inner_dim: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, inner_dim, bias=False)
        self.v = nn.Linear(d_model, inner_dim, bias=False)
        self.w2 = nn.Linear(inner_dim, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.v(x))


class TransformerBlock(nn.Module):
    def __init__(self, config: JuliaSLMConfig):
        super().__init__()
        self.ln1 = RMSNorm(config.d_model)
        self.attn = CausalSelfAttention(config.d_model, config.n_heads, config.head_dim)
        self.ln2 = RMSNorm(config.d_model)
        self.ffn = SwiGLUFFN(config.d_model, config.ffn_inner)

    def forward(self, x: torch.Tensor, rope: RotaryEmbedding,
                mask: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x), rope, mask)
        x = x + self.ffn(self.ln2(x))
        return x


class JuliaSLM(nn.Module):
    def __init__(self, config: JuliaSLMConfig):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.rope = RotaryEmbedding(config.head_dim, config.context_length, config.rope_base)
        self.blocks = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config.n_layers)]
        )
        self.ln_f = RMSNorm(config.d_model)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        mask = torch.triu(
            torch.full((T, T), float("-inf"), device=x.device, dtype=x.dtype),
            diagonal=1,
        )
        for block in self.blocks:
            x = block(x, self.rope, mask)
        x = self.ln_f(x)
        return F.linear(x, self.tok_emb.weight)


config = JuliaSLMConfig()
print(f"JuliaSLM: d={config.d_model}, L={config.n_layers}, H={config.n_heads}, "
      f"hd={config.head_dim}, ffn={config.ffn_inner}")

In [ ]:
# 6. Load JuliaSLM weights from NPZ + verify baseline

npz_data = np.load("juliaslm_weights.npz")

print("NPZ keys:")
for key in sorted(npz_data.files):
    arr = npz_data[key]
    print(f"  {key}: {arr.shape} {arr.dtype}")

model = JuliaSLM(config).to(device)

# NPZ key mapping: bare names -> .weight suffix for nn.Linear
model_keys = set(model.state_dict().keys())
state_dict = {}
for key in npz_data.files:
    if key.startswith("_hp_"):
        continue
    tensor = torch.from_numpy(npz_data[key].copy())
    if key in model_keys:
        state_dict[key] = tensor
    elif key + ".weight" in model_keys:
        state_dict[key + ".weight"] = tensor
    else:
        print(f"  WARNING: no match for NPZ key '{key}'")

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"\nLoaded {len(state_dict)} arrays")
rope_missing = [k for k in missing if "rope" in k or "cache" in k]
other_missing = [k for k in missing if k not in rope_missing]
if rope_missing:
    print(f"Missing (expected, RoPE buffers): {rope_missing}")
if other_missing:
    print(f"WARNING — Missing weights: {other_missing}")

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} params ({n_params/1e6:.2f}M)")


def evaluate_model(model, val_inputs, val_labels, batch_size=64):
    """Compute val loss and perplexity."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    dev = next(model.parameters()).device
    with torch.no_grad():
        for i in range(0, len(val_inputs), batch_size):
            batch_in = val_inputs[i:i+batch_size].to(dev)
            batch_tgt = val_labels[i:i+batch_size].to(dev)
            logits = model(batch_in)
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.float().reshape(B*T, V), batch_tgt.reshape(B*T), reduction="sum"
            )
            total_loss += loss.item()
            total_tokens += B * T
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl


print("\nEvaluating baseline...")
base_loss, base_ppl = evaluate_model(model, val_inputs, val_labels)
print(f"Baseline: val_loss={base_loss:.4f} ppl={base_ppl:.1f}")
print(f"Expected: val_loss=3.5403  ppl=34.5")

In [ ]:
# 7. SVD diagnostics — per-layer analysis
#
# This is the wolves' core signal: each layer has a different SVD spectrum.
# Layers with fast energy decay are highly compressible (low rank needed).
# Layers with slow decay need more rank to preserve information.

def diagnose_layers(model):
    """Analyze each Linear layer's SVD spectrum at multiple energy thresholds."""
    results = []
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        w = module.weight.detach().cpu()
        out_dim, in_dim = w.shape
        full_rank = min(out_dim, in_dim)

        s = torch.linalg.svdvals(w)
        total_energy = (s ** 2).sum().item()
        cum_energy = torch.cumsum(s ** 2, dim=0)

        def rank_at(frac):
            return min(int((cum_energy < frac * total_energy).sum().item()) + 1, full_rank)

        results.append({
            "name": name,
            "shape": (out_dim, in_dim),
            "full_rank": full_rank,
            "rank_80": rank_at(0.80),
            "rank_90": rank_at(0.90),
            "rank_95": rank_at(0.95),
            "rank_99": rank_at(0.99),
            "params": out_dim * in_dim,
        })
    return results


diag = diagnose_layers(model)

print(f"{'Layer':<40} {'Shape':>12} {'Full':>5} {'R80':>5} {'R90':>5} {'R95':>5} {'R99':>5}")
print("-" * 85)
for d in diag:
    print(f"{d['name']:<40} {str(d['shape']):>12} {d['full_rank']:>5} "
          f"{d['rank_80']:>5} {d['rank_90']:>5} {d['rank_95']:>5} {d['rank_99']:>5}")

# Compute param savings at each threshold
print(f"\n{'Threshold':<12} {'SVD Params':>12} {'Original':>12} {'Savings':>10}")
print("-" * 48)
total_orig = sum(d["params"] for d in diag)
for thresh, key in [("80%", "rank_80"), ("90%", "rank_90"), ("95%", "rank_95"), ("99%", "rank_99")]:
    svd_params = sum(d[key] * sum(d["shape"]) for d in diag)
    print(f"{thresh:<12} {svd_params:>12,} {total_orig:>12,} {(1-svd_params/total_orig)*100:>9.1f}%")

In [ ]:
# 8. FactoredLinear + compress infrastructure
#
# Same as wolves notebook: replace nn.Linear with SVD-factored A @ B.
# Params go from out*in to k*(out+in) where k = target rank.

class FactoredLinear(nn.Module):
    """SVD-compressed linear: W ≈ A @ B where A=(out,k), B=(k,in)."""

    def __init__(self, A: torch.Tensor, B: torch.Tensor):
        super().__init__()
        self.A = nn.Parameter(A)  # (out, k)
        self.B = nn.Parameter(B)  # (k, in)

    @property
    def weight(self):
        return self.A @ self.B

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(F.linear(x, self.B), self.A)


def compress_layer(linear: nn.Linear, target_rank: int) -> nn.Module:
    """Replace nn.Linear with FactoredLinear via SVD truncation."""
    w = linear.weight.detach()
    full_rank = min(w.shape)
    k = min(target_rank, full_rank)
    if k >= full_rank:
        return linear

    U, S, Vh = torch.linalg.svd(w, full_matrices=False)
    sqrt_S = S[:k].sqrt()
    A = U[:, :k] * sqrt_S.unsqueeze(0)    # (out, k)
    B = sqrt_S.unsqueeze(1) * Vh[:k, :]    # (k, in)
    return FactoredLinear(A, B)


def compress_model(model, rank_schedule: Dict[str, int]):
    """Create compressed copy with per-layer SVD rank targets."""
    compressed = copy.deepcopy(model)
    for name, target_rank in rank_schedule.items():
        parts = name.split(".")
        parent = compressed
        for part in parts[:-1]:
            parent = getattr(parent, part)
        attr = parts[-1]
        linear = getattr(parent, attr)
        if isinstance(linear, nn.Linear):
            factored = compress_layer(linear, target_rank)
            setattr(parent, attr, factored)
    return compressed


layer_names = [name for name, m in model.named_modules() if isinstance(m, nn.Linear)]
print(f"Found {len(layer_names)} Linear layers to compress")

In [ ]:
# 9. Build SVD-guided rank schedules + create compressed models
#
# Instead of random wolves schedules, use the deterministic SVD energy
# analysis to set per-layer ranks. Each layer gets exactly the rank
# needed to retain the target energy fraction.

def build_svd_schedule(diag_info, energy_key):
    """Build rank schedule from SVD diagnostics at given energy threshold."""
    schedule = {}
    for d in diag_info:
        schedule[d["name"]] = d[energy_key]
    return schedule


svd_configs = {
    "SVD-95": build_svd_schedule(diag, "rank_95"),
    "SVD-90": build_svd_schedule(diag, "rank_90"),
    "SVD-80": build_svd_schedule(diag, "rank_80"),
}

compressed_models = {}
pre_ft_results = {}

print(f"{'Config':<10} {'Params':>10} {'Reduction':>10} {'Pre-FT Loss':>12} {'Pre-FT PPL':>11}")
print("-" * 55)

for name, schedule in svd_configs.items():
    # Print per-layer ranks for this config
    ranks = list(schedule.values())
    print(f"\n{name} rank range: [{min(ranks)}-{max(ranks)}]")
    
    model_c = compress_model(model, schedule)
    model_c = model_c.to(device)
    comp_params = sum(p.numel() for p in model_c.parameters())
    reduction = 1 - comp_params / n_params
    
    loss, ppl = evaluate_model(model_c, val_inputs, val_labels)
    pre_ft_results[name] = {"params": comp_params, "reduction": reduction,
                             "pre_loss": loss, "pre_ppl": ppl,
                             "schedule": schedule}
    compressed_models[name] = model_c
    
    print(f"{name:<10} {comp_params:>10,} {reduction:>9.1%} {loss:>12.4f} {ppl:>11.1f}")

print(f"\n{'Source':<10} {n_params:>10,} {'---':>10} {base_loss:>12.4f} {base_ppl:>11.1f}")
print(f"\nPre-fine-tune losses show information lost by SVD truncation.")
print(f"Fine-tuning recovers much of this — the factored weights can learn.")

In [ ]:
# 10. Fine-tune function

def finetune(model, config_name, schedule, train_inputs, train_labels,
             val_inputs, val_labels, n_steps=2000, lr=6e-4, batch_size=64,
             warmup=200, eval_every=250):
    """Fine-tune a compressed model. Returns (model, best_loss, best_ppl, history)."""
    
    comp_params = sum(p.numel() for p in model.parameters())
    pre_loss, pre_ppl = evaluate_model(model, val_inputs, val_labels)
    ranks = list(schedule.values())
    
    run = wandb.init(
        project="symbiogenesis",
        name=f"compress-svd-juliaslm-{config_name}",
        config={
            "method": "svd_guided_compression",
            "source": "JuliaSLM (5.04M, d=256, 6L, val_loss=3.54)",
            "config_name": config_name,
            "n_params": comp_params,
            "rank_min": min(ranks),
            "rank_max": max(ranks),
            "rank_mean": sum(ranks) / len(ranks),
            "pre_ft_loss": pre_loss,
            "n_steps": n_steps,
            "lr": lr,
            "batch_size": batch_size,
        },
        tags=["compression", "svd-guided", "juliaslm", "symbiogenesis", config_name],
        reinit="finish_previous",
    )
    # Log per-layer rank schedule
    wandb.config.update({"rank_schedule": schedule})
    
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=0.1, betas=(0.9, 0.95)
    )
    min_lr = lr * 0.1
    
    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / max(warmup, 1)
        progress = (step - warmup) / max(n_steps - warmup, 1)
        return max(min_lr / lr, 0.5 * (1.0 + math.cos(math.pi * progress)))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))
    
    n_train = len(train_inputs)
    model.train()
    best_loss = float("inf")
    best_ppl = float("inf")
    history = []
    t_start = time.time()
    step = 0
    
    print(f"\n{'='*60}")
    print(f"Fine-tuning {config_name}: {comp_params:,} params, {n_steps} steps")
    print(f"Pre-FT: loss={pre_loss:.4f} ppl={pre_ppl:.1f}")
    print(f"Ranks: [{min(ranks)}-{max(ranks)}], mean={sum(ranks)/len(ranks):.0f}")
    print(f"Precision: {amp_dtype}, batch={batch_size}, lr={lr}")
    print(f"{'='*60}")
    
    while step < n_steps:
        perm = torch.randperm(n_train)
        for i in range(0, n_train, batch_size):
            if step >= n_steps:
                break
            
            idx = perm[i:i+batch_size]
            batch_in = train_inputs[idx].to(device)
            batch_tgt = train_labels[idx].to(device)
            
            with torch.amp.autocast("cuda", enabled=True, dtype=amp_dtype):
                logits = model(batch_in)
                B, T, V = logits.shape
                loss = F.cross_entropy(logits.reshape(B*T, V), batch_tgt.reshape(B*T))
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
            if step % 50 == 0:
                wandb.log({"train/loss": loss.item(),
                           "train/lr": scheduler.get_last_lr()[0]}, step=step)
            
            if step > 0 and step % eval_every == 0:
                val_loss, val_ppl = evaluate_model(model, val_inputs, val_labels)
                history.append((step, val_loss, val_ppl))
                wandb.log({"val/loss": val_loss, "val/ppl": val_ppl}, step=step)
                marker = " ** BEST **" if val_loss < best_loss else ""
                if val_loss < best_loss:
                    best_loss = val_loss
                    best_ppl = val_ppl
                    torch.save(model.state_dict(), f"svd_{config_name}_best.pt")
                elapsed = time.time() - t_start
                print(f"  [{config_name} step {step:5d}] val={val_loss:.4f} ppl={val_ppl:.1f} "
                      f"lr={scheduler.get_last_lr()[0]:.2e} ({elapsed:.0f}s){marker}")
                model.train()
            
            step += 1
    
    # Final eval
    final_loss, final_ppl = evaluate_model(model, val_inputs, val_labels)
    history.append((step, final_loss, final_ppl))
    if final_loss < best_loss:
        best_loss = final_loss
        best_ppl = final_ppl
        torch.save(model.state_dict(), f"svd_{config_name}_best.pt")
    
    elapsed = time.time() - t_start
    wandb.log({"val/final_loss": best_loss, "val/final_ppl": best_ppl})
    print(f"  {config_name} done ({elapsed:.0f}s): best val_loss={best_loss:.4f} ppl={best_ppl:.1f}")
    wandb.finish()
    
    # Reload best checkpoint
    best_sd = torch.load(f"svd_{config_name}_best.pt", map_location=device, weights_only=True)
    model.load_state_dict(best_sd, strict=False)
    model.eval()
    
    return model, best_loss, best_ppl, history


print("finetune() defined.")

In [ ]:
# 11. Fine-tune all 3 SVD-compressed models sequentially

post_ft_results = {}

for name in ["SVD-95", "SVD-90", "SVD-80"]:
    model_c = compressed_models[name]
    schedule = svd_configs[name]
    
    model_c, best_loss, best_ppl, history = finetune(
        model_c, name, schedule,
        train_inputs, train_labels,
        val_inputs, val_labels,
        n_steps=2000, lr=6e-4, batch_size=64,
        warmup=200, eval_every=250,
    )
    
    post_ft_results[name] = {
        "best_loss": best_loss,
        "best_ppl": best_ppl,
        "history": history,
    }
    compressed_models[name] = model_c

print("\nAll fine-tuning complete!")

In [ ]:
# 12. Results comparison

print("\n" + "=" * 80)
print("SVD-GUIDED COMPRESSION RESULTS — JuliaSLM")
print("=" * 80)

print(f"\n{'Config':<10} {'Params':>10} {'Reduc':>7} {'Pre-FT':>9} {'Post-FT':>9} {'Recovery':>10} {'PPL':>8}")
print("-" * 65)

for name in ["SVD-95", "SVD-90", "SVD-80"]:
    pre = pre_ft_results[name]
    post = post_ft_results[name]
    recovery = pre["pre_loss"] - post["best_loss"]
    print(f"{name:<10} {pre['params']:>10,} {pre['reduction']:>6.1%} "
          f"{pre['pre_loss']:>9.4f} {post['best_loss']:>9.4f} "
          f"{recovery:>+9.4f} {post['best_ppl']:>8.1f}")

print(f"{'Source':<10} {n_params:>10,} {'---':>7} "
      f"{base_loss:>9.4f} {'---':>9} {'---':>10} {base_ppl:>8.1f}")

# Per-layer rank details for best config
best_name = min(post_ft_results, key=lambda n: post_ft_results[n]["best_loss"])
best_schedule = svd_configs[best_name]
print(f"\nBest config: {best_name}")
print(f"\n{'Layer':<40} {'Rank':>6} {'Full':>6} {'Ratio':>7}")
print("-" * 60)
diag_map = {d['name']: d for d in diag}
for name_l, rank in best_schedule.items():
    d = diag_map[name_l]
    print(f"{name_l:<40} {rank:>6} {d['full_rank']:>6} {rank/d['full_rank']:>6.0%}")

# Scaling law context
print(f"\n{'='*80}")
print("SCALING LAW CONTEXT (curated data, BPE vocab=2000, ctx=256)")
print(f"{'='*80}")
print(f"\n{'Model':<35} {'Params':>10} {'Val Loss':>10} {'PPL':>8}")
print("-" * 65)

for name in ["SVD-80", "SVD-90", "SVD-95"]:
    pre = pre_ft_results[name]
    post = post_ft_results[name]
    print(f"{'JuliaSLM-'+name:<35} {pre['params']/1e6:>9.2f}M {post['best_loss']:>10.4f} {post['best_ppl']:>8.1f}")

print(f"{'SymbioSLM':<35} {'4.07M':>10} {'3.6200':>10} {'37.3':>8}")
print(f"{'MonarchSLM':<35} {'4.98M':>10} {'3.6500':>10} {'38.4':>8}")
print(f"{'JuliaSLM (source)':<35} {'5.04M':>10} {base_loss:>10.4f} {base_ppl:>8.1f}")
print(f"{'SymbioGPT-10M':<35} {'11.05M':>10} {'3.5630':>10} {'35.3':>8}")

best_post = post_ft_results[best_name]
best_pre = pre_ft_results[best_name]
print(f"\nBest: {best_name} ({best_pre['params']/1e6:.2f}M) "
      f"val_loss={best_post['best_loss']:.4f} ppl={best_post['best_ppl']:.1f} "
      f"({best_pre['reduction']:.0%} smaller)")

In [ ]:
# 13. Upload to HuggingFace
from huggingface_hub import HfApi, create_repo
import json

best_name = min(post_ft_results, key=lambda n: post_ft_results[n]["best_loss"])

COMPRESSED_REPO = "LisaMegaWatts/JuliaSLM-compressed-svd"

# Save metadata
os.makedirs("compressed_svd", exist_ok=True)
meta_path = "compressed_svd/svd_compression_metadata.json"
all_results = {}
for name in ["SVD-95", "SVD-90", "SVD-80"]:
    pre = pre_ft_results[name]
    post = post_ft_results[name]
    all_results[name] = {
        "rank_schedule": pre["schedule"],
        "params": pre["params"],
        "reduction": pre["reduction"],
        "pre_finetune_loss": pre["pre_loss"],
        "post_finetune_loss": post["best_loss"],
        "post_finetune_ppl": post["best_ppl"],
    }

with open(meta_path, "w") as f:
    json.dump({
        "method": "svd_guided_compression",
        "description": "Per-layer SVD rank targets from energy analysis (wolves signal without evolution)",
        "source_model": "LisaMegaWatts/JuliaSLM",
        "source_params": n_params,
        "source_loss": base_loss,
        "best_config": best_name,
        "finetune_steps": 2000,
        "finetune_lr": 6e-4,
        "configs": all_results,
    }, f, indent=2)

print(f"Saved metadata: {meta_path}")

# Upload
hf_api = HfApi()
try:
    create_repo(COMPRESSED_REPO, exist_ok=True)
    
    for name in ["SVD-95", "SVD-90", "SVD-80"]:
        ckpt = f"svd_{name}_best.pt"
        if os.path.exists(ckpt):
            size_mb = os.path.getsize(ckpt) / 1e6
            post = post_ft_results[name]
            print(f"Uploading {ckpt} ({size_mb:.1f} MB, loss={post['best_loss']:.4f})...")
            hf_api.upload_file(
                path_or_fileobj=ckpt,
                path_in_repo=ckpt,
                repo_id=COMPRESSED_REPO,
                commit_message=f"{name}: val_loss={post['best_loss']:.4f} ppl={post['best_ppl']:.1f}",
            )
    
    hf_api.upload_file(
        path_or_fileobj=meta_path,
        path_in_repo="svd_compression_metadata.json",
        repo_id=COMPRESSED_REPO,
        commit_message="SVD compression metadata and per-layer rank schedules",
    )
    
    print(f"\nUploaded to: https://huggingface.co/{COMPRESSED_REPO}")
except Exception as e:
    print(f"HF upload failed: {e}")

print("\nDone!")